# Full results analysis: patient-cluster uncertainty and operating-point transfer

This Colab notebook performs the post-training analysis for the chest radiograph transport study from saved per-image predictions. It does **not** train models or run image inference. The public scripts in `src/` describe model training and transfer evaluation; the required datasets must be obtained from their original providers.

The notebook checks all 16 source-to-target comparisons and five seeds, computes per-seed metrics and five-seed ensemble metrics, estimates 500-replicate target-set bootstrap intervals, applies source-validation Youden thresholds without tuning on target labels, and writes aggregate tables and heatmaps. Patient identifiers are used for bootstrap clusters when available; Kaggle/Kermany has no usable patient identifier in this project, so image-level resampling is used when it is the target. These intervals do not measure site-to-site or prospective clinical performance. No microbiology outcomes are included.

Before running, configure `PROJECT` in the next cell to point to your own Drive folder. The expected file layout and columns are described in `paper_release/README.md`.

In [ ]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, log_loss
)
from google.colab import drive

drive.mount("/content/drive")

# Change this to the Google Drive folder containing your own generated predictions.
PROJECT = Path("/content/drive/MyDrive/chest_xray_project")
PRED_DIR = PROJECT / "predictions" / "cross_dataset"
OPERATING_POINTS = PROJECT / "source_validation_operating_points_seedwise.csv"
OUTPUT_DIR = Path("/content/drive/MyDrive/chest_xray_analysis_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = ["Kaggle_Pneumonia", "RSNA_LungOpacity", "CheXpert_Pneumonia", "CheXpert_LungOpacity"]
TARGETS = ["Kaggle_Pneumonia", "RSNA_LungOpacity", "CheXpert_Pneumonia", "CheXpert_LungOpacity"]
EXPECTED_SEEDS = [42, 1337, 2025, 7, 99]
BOOTSTRAP_REPLICATES = 500
RANDOM_SEED = 20261002

assert PRED_DIR.is_dir(), f"Prediction folder not found: {PRED_DIR}"
assert OPERATING_POINTS.is_file(), f"Source-validation thresholds not found: {OPERATING_POINTS}"
print(f"Predictions: {PRED_DIR}")
print(f"Results will be saved to: {OUTPUT_DIR}")
print(f"Patient-cluster bootstrap replicates per comparison: {BOOTSTRAP_REPLICATES}")

In [ ]:
def ece15(y, p):
    # Expected calibration error using 15 equal-width bins.
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    edges = np.linspace(0.0, 1.0, 16)
    total = len(y)
    value = 0.0
    for i in range(15):
        if i < 14:
            mask = (p >= edges[i]) & (p < edges[i + 1])
        else:
            mask = (p >= edges[i]) & (p <= edges[i + 1])
        if mask.any():
            value += mask.sum() / total * abs(y[mask].mean() - p[mask].mean())
    return float(value)

def metric_set(y, p, threshold=0.5):
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-7, 1 - 1e-7)
    pred05 = (p >= 0.5).astype(int)
    confidence = np.maximum(p, 1 - p)
    wrong = pred05 != y
    covered = confidence >= 0.90
    return {
        "prevalence": float(y.mean()),
        "auroc": float(roc_auc_score(y, p)),
        "auprc": float(average_precision_score(y, p)),
        "balanced_accuracy_at_0p5": float(balanced_accuracy_score(y, pred05)),
        "brier": float(brier_score_loss(y, p)),
        "ece15": ece15(y, p),
        "nll": float(log_loss(y, p, labels=[0, 1])),
        "high_confidence_coverage": float(covered.mean()),
        "hcer_all_at_0p90": float(np.mean(wrong & covered)),
        "hcer_covered_at_0p90": float(np.mean(wrong[covered])) if covered.any() else np.nan,
        "flags_per_1000": float(np.mean(p >= threshold) * 1000),
    }

def seed_t_interval(values):
    values = np.asarray(values, dtype=float)
    mean = float(np.mean(values))
    if len(values) < 2:
        return mean, np.nan, np.nan, np.nan
    sd = float(np.std(values, ddof=1))
    half = float(stats.t.ppf(0.975, df=len(values)-1) * sd / np.sqrt(len(values)))
    return mean, sd, mean-half, mean+half

# Use only saved source-validation Youden thresholds; no target label enters threshold choice.
op = pd.read_csv(OPERATING_POINTS)
op = op[(op["threshold_policy"] == "source_youden") & (op["source_task"].isin(SOURCES))]
assert set(op["source_task"]) == set(SOURCES), "Missing a source-validation Youden threshold."
thresholds = op.groupby("source_task")["threshold"].mean().to_dict()
threshold_table = op[["source_task", "seed", "threshold", "selection_rule"]].copy()
threshold_table.to_csv(OUTPUT_DIR / "source_youden_thresholds_by_seed.csv", index=False)
print("Mean source-validation Youden thresholds:", {k: round(v, 4) for k, v in thresholds.items()})

In [ ]:
def load_pair(source, target):
    pattern = f"{source}__to__{target}__seed_*.csv"
    files = sorted(PRED_DIR.glob(pattern))
    seed_frames = []
    for path in files:
        raw = pd.read_csv(path, usecols=[
            "source_task", "target_task", "seed", "target_subset", "image_id",
            "patient_id", "true_label", "pred_prob"
        ])
        raw = raw[raw["target_subset"] == "test_split"].copy()
        if raw.empty:
            continue
        seeds = raw["seed"].dropna().astype(int).unique()
        if len(seeds) != 1:
            raise ValueError(f"Expected one seed in {path.name}; found {seeds}")
        seed = int(seeds[0])
        if raw["image_id"].duplicated().any():
            raise ValueError(f"Duplicate image_id rows in {path.name}")
        part = raw[["image_id", "patient_id", "true_label", "pred_prob"]].copy()
        part["true_label"] = part["true_label"].astype(int)
        part = part.rename(columns={"pred_prob": f"p_{seed}"})
        seed_frames.append((seed, part))
    found = sorted(seed for seed, _ in seed_frames)
    if found != sorted(EXPECTED_SEEDS):
        raise ValueError(f"{source} -> {target}: expected seeds {EXPECTED_SEEDS}, found {found}")

    merged = seed_frames[0][1]
    for _, frame in seed_frames[1:]:
        merged = merged.merge(
            frame, on=["image_id", "patient_id", "true_label"],
            how="outer", validate="one_to_one", indicator=True
        )
        if not (merged["_merge"] == "both").all():
            raise ValueError(f"Image or label mismatch across seeds: {source} -> {target}")
        merged = merged.drop(columns="_merge")
    prob_cols = [f"p_{s}" for s in EXPECTED_SEEDS]
    if merged[prob_cols].isna().any().any():
        raise ValueError(f"Missing saved predictions: {source} -> {target}")
    merged["ensemble_probability"] = merged[prob_cols].mean(axis=1)
    patient = merged["patient_id"].fillna("").astype(str).str.strip()
    has_patient = patient.ne("")
    merged["bootstrap_cluster"] = np.where(
        has_patient, "patient:" + patient, "image:" + merged["image_id"].astype(str)
    )
    return merged, bool(has_patient.all()), len(files)

def bootstrap_pair(data, threshold, rng, B=BOOTSTRAP_REPLICATES):
    y = data["true_label"].to_numpy(dtype=int)
    p = data["ensemble_probability"].to_numpy(dtype=float)
    point = metric_set(y, p, threshold)
    groups = data["bootstrap_cluster"].to_numpy()
    group_names = pd.unique(groups)
    row_indices = {g: np.flatnonzero(groups == g) for g in group_names}
    boot = {k: [] for k in point}
    attempts = 0
    while len(boot["auroc"]) < B:
        attempts += 1
        sampled_groups = rng.choice(group_names, size=len(group_names), replace=True)
        idx = np.concatenate([row_indices[g] for g in sampled_groups])
        if np.unique(y[idx]).size < 2:
            if attempts > B * 10:
                raise RuntimeError("Too many one-class bootstrap draws; inspect class/group distribution.")
            continue
        vals = metric_set(y[idx], p[idx], threshold)
        for key, value in vals.items():
            boot[key].append(value)
        if attempts > B * 10:
            raise RuntimeError("Bootstrap exceeded retry limit.")
    intervals = {
        key: (float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975)))
        for key, values in boot.items()
    }
    return point, intervals

started = time.time()
rng = np.random.default_rng(RANDOM_SEED)
seedwise_rows, seed_summary_rows, bootstrap_rows, audit_rows = [], [], [], []

for source in SOURCES:
    for target in TARGETS:
        pair, patient_ids_used, file_count = load_pair(source, target)
        y = pair["true_label"].to_numpy(dtype=int)
        threshold = float(thresholds[source])
        per_seed = {}
        for seed in EXPECTED_SEEDS:
            values = metric_set(y, pair[f"p_{seed}"].to_numpy(dtype=float), threshold)
            per_seed[seed] = values
            seedwise_rows.append({
                "source_task": source, "target_task": target, "seed": seed,
                "n_images": len(pair), "n_positive": int(y.sum()),
                "n_negative": int(len(y)-y.sum()), **values
            })
        for metric in per_seed[EXPECTED_SEEDS[0]]:
            mean, sd, lo, hi = seed_t_interval([per_seed[s][metric] for s in EXPECTED_SEEDS])
            seed_summary_rows.append({
                "source_task": source, "target_task": target, "metric": metric,
                "seed_mean": mean, "seed_sd": sd, "seed_t95_low": lo, "seed_t95_high": hi,
                "n_seeds": len(EXPECTED_SEEDS)
            })
        point, intervals = bootstrap_pair(pair, threshold, rng)
        bootstrap_rows.append({
            "source_task": source, "target_task": target,
            "n_images": len(pair), "n_positive": int(y.sum()), "n_negative": int(len(y)-y.sum()),
            "n_bootstrap_clusters": int(pair["bootstrap_cluster"].nunique()),
            "bootstrap_unit": "patient" if patient_ids_used else "image (patient ID unavailable)",
            "n_seed_ensemble_members": len(EXPECTED_SEEDS), "source_youden_threshold_mean": threshold,
            **{f"ensemble_{k}": v for k, v in point.items()},
            **{f"{k}_bootstrap95_low": v[0] for k, v in intervals.items()},
            **{f"{k}_bootstrap95_high": v[1] for k, v in intervals.items()},
        })
        audit_rows.append({
            "source_task": source, "target_task": target, "files_found": file_count,
            "seeds_found": ",".join(map(str, EXPECTED_SEEDS)), "n_test_images": len(pair),
            "positive_images": int(y.sum()), "negative_images": int(len(y)-y.sum()),
            "positive_prevalence": float(y.mean()),
            "patient_ids_available_for_all_images": patient_ids_used,
            "bootstrap_cluster_count": int(pair["bootstrap_cluster"].nunique()),
            "target_subset": "test_split", "threshold_source": "source validation only"
        })
        print(f"Finished {source} -> {target}: n={len(pair):,}, groups={pair['bootstrap_cluster'].nunique():,}")

seedwise = pd.DataFrame(seedwise_rows)
seed_summary = pd.DataFrame(seed_summary_rows)
bootstrap_results = pd.DataFrame(bootstrap_rows)
audit = pd.DataFrame(audit_rows)

expected_pairs = len(SOURCES) * len(TARGETS)
assert len(bootstrap_results) == expected_pairs
assert audit["files_found"].eq(len(EXPECTED_SEEDS)).all()
assert audit["target_subset"].eq("test_split").all()

seedwise.to_csv(OUTPUT_DIR / "primary_test_metrics_by_seed.csv", index=False)
seed_summary.to_csv(OUTPUT_DIR / "primary_seed_mean_t_intervals.csv", index=False)
bootstrap_results.to_csv(OUTPUT_DIR / "primary_patient_cluster_bootstrap_ensemble.csv", index=False)
audit.to_csv(OUTPUT_DIR / "prediction_and_split_audit.csv", index=False)
print(f"All {expected_pairs} comparisons complete in {(time.time()-started)/60:.1f} minutes.")

In [ ]:
# Four complete 4 x 4 views; each cell is one trained source evaluated on one held-out target.
short = {
    "Kaggle_Pneumonia": "Kaggle pneumonia",
    "RSNA_LungOpacity": "RSNA opacity",
    "CheXpert_Pneumonia": "CheXpert pneumonia",
    "CheXpert_LungOpacity": "CheXpert opacity",
}
heatmap_specs = [
    ("ensemble_auroc", "Five-seed ensemble AUROC", "AUROC", "ensemble_auroc_heatmap.png"),
    ("ensemble_auprc", "Five-seed ensemble AUPRC", "AUPRC", "ensemble_auprc_heatmap.png"),
    ("ensemble_ece15", "Five-seed ensemble calibration error", "ECE15 (lower is better)", "ensemble_ece15_heatmap.png"),
    ("ensemble_flags_per_1000", "Flags at source-validation Youden threshold", "Images flagged per 1,000", "ensemble_flags_heatmap.png"),
]
for metric, title, colorbar_label, filename in heatmap_specs:
    matrix = bootstrap_results.pivot(index="target_task", columns="source_task", values=metric)
    matrix = matrix.reindex(index=TARGETS, columns=SOURCES)
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(matrix.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(SOURCES)), [short[s] for s in SOURCES], rotation=20, ha="right")
    ax.set_yticks(range(len(TARGETS)), [short[t] for t in TARGETS])
    ax.set_xlabel("Training task (source)")
    ax.set_ylabel("Held-out evaluation task (target)")
    ax.set_title(title)
    for i in range(len(TARGETS)):
        for j in range(len(SOURCES)):
            ax.text(j, i, f"{matrix.iloc[i, j]:.3f}" if "flags" not in metric else f"{matrix.iloc[i, j]:.1f}",
                    ha="center", va="center", color="white", fontsize=9)
    fig.colorbar(im, ax=ax, label=colorbar_label)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

manifest = {
    "analysis": "patient-cluster bootstrap for five-seed probability ensembles; per-seed metrics and Student-t intervals",
    "primary_source_tasks": SOURCES,
    "primary_target_tasks": TARGETS,
    "seeds": EXPECTED_SEEDS,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "comparisons": int(len(bootstrap_results)),
    "prediction_files_read": int(audit["files_found"].sum()),
    "cluster_policy": "patient_id when available; image_id fallback; Kaggle has no patient ID",
    "threshold_policy": "mean of five source-validation Youden thresholds per source task",
    "no_retraining": True,
    "no_image_inference": True,
    "microbiology_outcomes_present": False,
    "outputs": [p.name for p in OUTPUT_DIR.iterdir() if p.is_file()],
}
(OUTPUT_DIR / "analysis_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print("\nBootstrap table preview:")
display(bootstrap_results[[
    "source_task", "target_task", "n_images", "n_bootstrap_clusters",
    "ensemble_auroc", "auroc_bootstrap95_low", "auroc_bootstrap95_high",
    "ensemble_auprc", "ensemble_ece15", "ensemble_flags_per_1000"
]].round(3))
print(f"\nSaved all outputs to: {OUTPUT_DIR}")